Environment set up and importing libraries



In [ ]:
# Install and import required libraries
!pip install yfinance scikit-learn seaborn matplotlib pandas numpy scipy -q

In [ ]:
import os
import datetime
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, cut_tree
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.autolayout'] = True
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
print("Libraries imported successfully.")

Libraries imported successfully.


1.1 Ticker extraction and data download

In [ ]:
# 1.1 Ticker Extraction from Wikipedia
retrieval_date = datetime.date.today().isoformat()
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

print(f"Extracting S&P 500 constituents from Wikipedia on: {retrieval_date}")
tables = pd.read_html(url, storage_options=headers)
sp500_meta = tables[0][['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry']].copy()

Extracting S&P 500 constituents from Wikipedia on: 2026-11-02


In [ ]:
# Adjust ticker symbols for Yahoo Finance (e.g. BRK.B -> BRK-B)
sp500_meta['Ticker'] = sp500_meta['Symbol'].str.replace('.', '-', regex=False)
tickers = sp500_meta['Ticker'].tolist()
print(f"Total constituents extracted: {len(tickers)}")

Total constituents extracted: 503


In [ ]:
# 1.1 Download Daily Historical Prices (2021-01-01 to 2025-01-01 exclusive)
start_date = '2021-01-01'
end_date = '2025-01-01'
all_symbols = tickers + ['^GSPC']

print(f"Downloading daily market data for {len(all_symbols)} symbols...")
# auto_adjust=False preserves both raw Close and dividend/split Adjusted Close
raw_data = yf.download(all_symbols, start=start_date, end=end_date, auto_adjust=False, group_by='column')

# Save local pickle snapshot for reproducibility & offline caching
raw_data.to_pickle("sp500_raw_market_data.pkl")
print(f"Download complete. Preserved snapshot shape: {raw_data.shape}")

[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['SNDK', 'HONA', 'FDXF', 'VYLR', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Download complete. Preserved snapshot shape: (1005, 3024)


1.2 Data validation

In [ ]:
import re

exclusion_log = []

# Benchmark reference dates
benchmark_adj = raw_data['Adj Close']['^GSPC'].dropna()
total_trading_days = len(benchmark_adj)
print(f"Benchmark (^GSPC) total valid trading sessions: {total_trading_days}")

# Pre-check 1: Duplicate tickers
dup_tickers = [t for t in tickers if tickers.count(t) > 1]
if dup_tickers:
    print(f"WARNING: Duplicate tickers detected and deduplicated: {set(dup_tickers)}")
tickers = list(dict.fromkeys(tickers))  # Remove duplicates, preserve order

# Pre-check 2: Ticker format (dot notation = Yahoo Finance will fail)
bad_format = [t for t in tickers if '.' in t]
if bad_format:
    print(f"WARNING: Dot-notation tickers found (should be '-'): {bad_format}")

Benchmark (^GSPC) total valid trading sessions: 1005


In [ ]:

valid_tickers = []

for ticker in tickers:
    try:
        close_s  = raw_data['Close'][ticker]
        adj_s    = raw_data['Adj Close'][ticker]
        vol_s    = raw_data['Volume'][ticker]

        # CHECK 1: Completeness — require >= 98% of benchmark trading days
        valid_days = adj_s.dropna().shape[0]
        if valid_days < (total_trading_days * 0.98):
            exclusion_log.append({
                'Ticker': ticker,
                'Reason': f'Incomplete history: {valid_days}/{total_trading_days} sessions'
            })
            continue

        # CHECK 2: Non-positive prices
        if (adj_s <= 0).any() or (close_s <= 0).any():
            exclusion_log.append({'Ticker': ticker, 'Reason': 'Non-positive prices encountered'})
            continue

        # CHECK 3: Negative volume
        if (vol_s < 0).any():
            exclusion_log.append({'Ticker': ticker, 'Reason': 'Negative volume records encountered'})
            continue
        # CHECK 4: Zero-volume days (> 2% of trading sessions = suspicious)
        # Shares with many zero-volume days have artificially low Frequency (F)
        zero_vol_count = (vol_s == 0).sum()
        zero_vol_pct   = zero_vol_count / total_trading_days
        if zero_vol_pct > 0.02:
            exclusion_log.append({
                'Ticker': ticker,
                'Reason': f'Excess zero-volume days: {zero_vol_count} ({zero_vol_pct*100:.1f}%)'
            })
            continue
        # CHECK 5: Extreme daily returns — flag splits / data errors
        # An unadjusted stock split appears as a sudden -50% or -90% move.
        # Adj Close should handle this, but data errors still occur.
        # Threshold: ±40% single-day move on Adj Close is flagged.
        daily_returns = adj_s.dropna().pct_change(fill_method=None)
        extreme_moves = (daily_returns.abs() > 0.40).sum()
        if extreme_moves > 0:
            max_move = daily_returns.abs().max()
            exclusion_log.append({
                'Ticker': ticker,
                'Reason': f'Extreme daily move detected: {extreme_moves} day(s) with |return| > 40% '
                          f'(max: {max_move*100:.1f}%). Possible unadjusted split or data error.'
            })
            continue

        # CHECK 6: Stale prices — long runs of identical closing price
        # 10+ consecutive identical Adj Close values = data stopped updating
        price_changes   = adj_s.dropna().diff().fillna(1)      # fillna(1) avoids flagging day 1
        unchanged       = (price_changes == 0)
        max_streak      = unchanged.groupby((unchanged != unchanged.shift()).cumsum()).cumsum().max()
        if max_streak >= 10:
            exclusion_log.append({
                'Ticker': ticker,
                'Reason': f'Stale price detected: {int(max_streak)} consecutive identical closing prices'
            })
            continue

        # CHECK 7: Gap handling for the remaining up to 2% missing days
        # We DROP the missing rows entirely (do NOT forward-fill).
        # Rationale: forward-filling prices creates artificial zero-return
        # days that dilute weekly return estimates and distort beta.
        # Missing days are documented here. Returns are computed on
        # actual trading days only (pct_change on the aligned series).
        missing_days = adj_s.isna().sum()
        if missing_days > 0:
            print(f"  [INFO] {ticker}: {missing_days} missing day(s) — rows dropped (not filled).")
            # No exclusion — within the 2% tolerance. Rows are dropped
            # automatically when we call .dropna() in later computations.

        valid_tickers.append(ticker)

    except Exception as e:
        exclusion_log.append({'Ticker': ticker, 'Reason': f'Extraction failure: {str(e)}'})

In [ ]:

# --- Save full exclusion log ---
exclusion_df = pd.DataFrame(exclusion_log)
exclusion_df.to_csv("exclusion_log.csv", index=False)

# --- Summary report ---
print(f"\n{'='*55}")
print(f"DATA VALIDATION SUMMARY")
print(f"{'='*55}")
print(f"  Total tickers examined:  {len(tickers)}")
print(f"  Retained valid shares:   {len(valid_tickers)}")
print(f"  Excluded shares:         {len(exclusion_df)}")
if len(exclusion_df) > 0:
    print(f"\n  Exclusion Breakdown:")
    for reason_prefix in ['Incomplete', 'Non-positive', 'Negative vol',
                          'Excess zero', 'Extreme daily', 'Stale price', 'Extraction']:
        count = exclusion_df['Reason'].str.startswith(reason_prefix).sum()
        if count > 0:
            print(f"    - {reason_prefix}...: {count} shares")
print(f"{'='*55}")

print("\nSample exclusions:")
display(exclusion_df.head(8))


DATA VALIDATION SUMMARY
  Total tickers examined:  503
  Retained valid shares:   481
  Excluded shares:         22

  Exclusion Breakdown:
    - Incomplete...: 16 shares
    - Excess zero...: 2 shares
    - Extreme daily...: 4 shares

Sample exclusions:


,Ticker,Reason
0,APP,Incomplete history: 935/1005 sessions
1,BE,Extreme daily move detected: 1 day(s) with |re...
2,CVNA,Extreme daily move detected: 5 day(s) with |re...
3,COIN,Incomplete history: 936/1005 sessions
4,CEG,Incomplete history: 742/1005 sessions
5,DXCM,Extreme daily move detected: 1 day(s) with |re...
6,EXE,Incomplete history: 979/1005 sessions
7,FDXF,Incomplete history: 0/1005 sessions


1.3 Weekly return and beta estimation - weekly

In [ ]:
# 1.3.1 Resample to Friday-ending weeks
adj_close_selected = raw_data['Adj Close'][valid_tickers + ['^GSPC']]

weekly_prices = adj_close_selected.resample('W-FRI').last()

# Drop the final row if data ends early in the week (Mon-Wed).
# When the market closes on a Friday (e.g. Good Friday), the last trading
# day is Thursday — last_close_date < last_week_date would be true even
# for a complete week, so we check weekday() instead.
last_close_date = adj_close_selected.dropna(how='all').index[-1]

if last_close_date.weekday() < 3:   # 0=Mon, 1=Tue, 2=Wed → genuinely partial
    print(f"[INFO] Partial final week detected — data ends {last_close_date.date()} "
          f"({last_close_date.day_name()}). Dropping last weekly row.")
    weekly_prices = weekly_prices.iloc[:-1]
else:
    print(f"[INFO] Final week is complete — data ends {last_close_date.date()} "
          f"({last_close_date.day_name()}). No row dropped.")

[INFO] Partial final week detected — data ends 2024-12-31 (Tuesday). Dropping last weekly row.


In [ ]:
# Compute weekly percentage returns.
# fill_method=None prevents silent forward-filling of missing prices.
weekly_returns = weekly_prices.pct_change(fill_method=None).iloc[1:]

market_weekly_returns = weekly_returns['^GSPC'].dropna()
print(f"Total complete weekly periods used: {len(market_weekly_returns)}")

# 1.3.2 Weekly Beta Estimation
results_rr    = []
rr_exclusions = []

for ticker in valid_tickers:
    try:
        share_ret = weekly_returns[ticker].dropna()

        # Align share and market returns on identical weekly timestamps
        paired = pd.concat(
            [share_ret.rename('share'), market_weekly_returns.rename('market')],
            axis=1
        ).dropna()

        n_paired = len(paired)

        if n_paired >= 52:
            # Compute variance over the same paired weeks as the covariance,
            # so both statistics are computed on identical observations.
            paired_market_var = paired['market'].var(ddof=1)

            if paired_market_var > 0:
                cov      = paired['share'].cov(paired['market'])
                beta     = cov / paired_market_var
                mean_ret = paired['share'].mean()

                results_rr.append({
                    'Ticker':             ticker,
                    'Mean_Weekly_Return': mean_ret,
                    'Weekly_Beta':        beta,
                    'Aligned_Weeks':      n_paired
                })
            else:
                rr_exclusions.append({
                    'Ticker': ticker,
                    'Reason': f'Zero market variance over {n_paired} paired weeks'
                })
        else:
            # Log shares that fall below the 52-week minimum rather than
            # discarding them silently, consistent with the exclusion policy.
            rr_exclusions.append({
                'Ticker': ticker,
                'Reason': f'Insufficient aligned weeks: {n_paired} (minimum 52 required)'
            })

    except Exception as e:
        rr_exclusions.append({'Ticker': ticker, 'Reason': f'Beta computation error: {str(e)}'})

# Append beta-stage exclusions to the master log and re-save
if rr_exclusions:
    rr_excl_df   = pd.DataFrame(rr_exclusions)
    exclusion_df = pd.concat([exclusion_df, rr_excl_df], ignore_index=True)
    exclusion_df.to_csv("exclusion_log.csv", index=False)
    print(f"[1.3] {len(rr_exclusions)} share(s) excluded at beta stage — appended to exclusion_log.csv")

risk_return_df = pd.DataFrame(results_rr).set_index('Ticker')

# Rebuild valid_tickers from shares that successfully produced a beta.
# Shares excluded at the beta stage are removed here so that the RFM
# calculation and clustering steps only operate on the common valid set.
valid_tickers = list(risk_return_df.index)
print(f"[1.3] Weekly Beta & Mean Return computed for {len(risk_return_df)} shares.")
print(f"[1.3] valid_tickers updated to {len(valid_tickers)} shares for downstream steps.")

Total complete weekly periods used: 207
[1.3] Weekly Beta & Mean Return computed for 481 shares.
[1.3] valid_tickers updated to 481 shares for downstream steps.


In [ ]:
# 1.3.3 Sanity Check
# Most betas for large-cap equities sit between 0.3 and 2.0.
# Mean weekly returns are typically small, around 0.1–0.5% per week.
# Values far outside these ranges are flagged for review but not removed,
# as extreme values can reflect genuine market behaviour (e.g. high-growth
# tech stocks) rather than data errors.
print("\n--- Sanity Check: Descriptive Statistics ---")
display(risk_return_df[['Mean_Weekly_Return', 'Weekly_Beta']].describe().round(5))

beta_outliers = risk_return_df[
    (risk_return_df['Weekly_Beta'] < 0) | (risk_return_df['Weekly_Beta'] > 3)
]
ret_outliers = risk_return_df[
    risk_return_df['Mean_Weekly_Return'].abs() > 0.02
]

if len(beta_outliers) > 0:
    print(f"\nBeta outliers flagged for review (beta < 0 or > 3): {len(beta_outliers)} share(s)")
    display(beta_outliers)
else:
    print("\nAll betas within expected range (0 – 3).")

if len(ret_outliers) > 0:
    print(f"\nReturn outliers flagged for review (|mean weekly return| > 2%): {len(ret_outliers)} share(s)")
    display(ret_outliers)
else:
    print("All mean weekly returns within expected range (±2%).")


--- Sanity Check: Descriptive Statistics ---


,Mean_Weekly_Return,Weekly_Beta
count,481.00000,481.00000
mean,0.00285,0.96428
std,0.00263,0.38355
min,-0.00447,0.05545
25%,0.00118,0.66658
50%,0.00262,0.94409
75%,0.00421,1.17484
max,0.01777,2.50452



All betas within expected range (0 – 3).
All mean weekly returns within expected range (±2%).


Task 1.4 | 1.5 - Adopted rfm calculation

In [ ]:
import numpy as np

# Use only dates where the S&P 500 index has a valid close (benchmark calendar)
common_dates  = raw_data['Close']['^GSPC'].dropna().index
obs_date      = common_dates[-1]          # last complete trading session
last_60_dates = common_dates[-60:]        # most recent 60 sessions

print(f"Observation date : {obs_date.strftime('%Y-%m-%d')}")
print(f"60-session window: {last_60_dates[0].strftime('%Y-%m-%d')}  →  "
      f"{last_60_dates[-1].strftime('%Y-%m-%d')}")
print(f"Shares entering RFM stage: {len(valid_tickers)}")

# RFM calculation function
def rfm_for(ticker):
    """
    Compute Recency, Frequency and Monetary for one share.

    Returns
    -------
    recency    : int   – calendar days since last high-volume session (NaN if none)
    frequency  : int   – count of high-volume sessions in the last 60 trading days
    monetary   : float – total traded value (Close × Volume) over the last 60 days
    event_dates: DatetimeIndex – all high-volume session dates (used by trace)

    Raises ValueError with a plain-English reason when the share cannot be scored.
    """
    close_s = raw_data['Close'][ticker].reindex(common_dates)
    vol_s   = raw_data['Volume'][ticker].reindex(common_dates)

    # Require at least 80 sessions (20 baseline warm-up + 60 scoring window)
    if vol_s.notna().sum() < 80:
        raise ValueError(
            f"Only {vol_s.notna().sum()} usable sessions — need ≥ 80")

    # The 60-session window must be complete for both price and volume
    if close_s.loc[last_60_dates].isna().any() or vol_s.loc[last_60_dates].isna().any():
        raise ValueError("Missing price or volume inside the 60-session window")

    # Rolling 20-session average volume (preceding sessions only, not today)
    baseline    = vol_s.shift(1).rolling(window=20).mean()
    is_high_vol = vol_s > 2.0 * baseline      # True where today > 2× baseline

    event_dates = vol_s[is_high_vol].index    # all high-volume session dates

    # Recency: calendar days since the most recent high-volume session
    recency   = (obs_date - event_dates[-1]).days if len(event_dates) > 0 else np.nan

    # Frequency: high-volume sessions inside the 60-day scoring window
    frequency = int(is_high_vol.loc[last_60_dates].sum())

    # Monetary: sum of (Close × Volume) across the 60-day scoring window
    monetary  = float(
        (close_s.loc[last_60_dates] * vol_s.loc[last_60_dates]).sum())

    return recency, frequency, monetary, event_dates

Observation date : 2024-12-31
60-session window: 2024-10-07  →  2024-12-31
Shares entering RFM stage: 481


In [ ]:

# Main RFM loop
rfm_records    = []
rfm_exclusions = []

for ticker in valid_tickers:
    try:
        r, f, m, _ = rfm_for(ticker)
        rfm_records.append({'Ticker': ticker, 'Recency': r,
                             'Frequency': f, 'Monetary': m})
    except Exception as exc:
        rfm_exclusions.append({'Ticker': ticker, 'Reason': f'RFM stage: {exc}'})

rfm_df = pd.DataFrame(rfm_records).set_index('Ticker')
print(f"\nAdapted RFM calculated for {len(rfm_df)} shares.")

# Append any RFM-stage exclusions to the master log
if rfm_exclusions:
    exclusion_df = pd.concat(
        [exclusion_df, pd.DataFrame(rfm_exclusions)], ignore_index=True)
    exclusion_df.to_csv("exclusion_log.csv", index=False)
    print(f"{len(rfm_exclusions)} share(s) excluded at RFM stage — "
          "appended to exclusion_log.csv")

# Shares with no qualifying high-volume event at any point in the study period
n_nan_r = rfm_df['Recency'].isna().sum()
print(f"Shares with no qualifying event (Recency = NaN): {n_nan_r}")

# Sanity check: distributions should be sensible
print("\nRFM distribution summary:")
print(rfm_df.describe().round(2))

# Flag any shares with Frequency = 0 inside the window (possible data issue)
zero_freq = rfm_df[rfm_df['Frequency'] == 0]
if not zero_freq.empty:
    print(f"\nShares with Frequency = 0 in the 60-session window ({len(zero_freq)}):")
    print(zero_freq.index.tolist())


Adapted RFM calculated for 481 shares.
Shares with no qualifying event (Recency = NaN): 0

RFM distribution summary:
       Recency  Frequency      Monetary
count   481.00     481.00  4.810000e+02
mean     20.35       2.98  3.963774e+10
std      21.41       1.33  1.297838e+11
min       4.00       0.00  1.297483e+09
25%      11.00       2.00  1.025033e+10
50%      11.00       3.00  1.670905e+10
75%      14.00       4.00  3.236510e+10
max     222.00       8.00  1.847697e+12

Shares with Frequency = 0 in the 60-session window (10):
['AZO', 'COR', 'CB', 'DUK', 'FCX', 'HRL', 'MKC', 'MDT', 'NVDA', 'WEC']


In [ ]:
nan_r = rfm_df[rfm_df['Recency'].isna()].index
if len(nan_r) > 0:
    exclusion_df = pd.concat([exclusion_df, pd.DataFrame(
        {'Ticker': nan_r, 'Reason': 'No qualifying high-volume event (Recency = NaN)'})], ignore_index=True)
    exclusion_df.to_csv("exclusion_log.csv", index=False)
    rfm_df = rfm_df.dropna(subset=['Recency'])

In [ ]:
trace_ticker = 'AAPL'

print("\n" + "=" * 65)
print(f"TASK 1.5  WORKED NUMERICAL TRACE — {trace_ticker}")
print("=" * 65)

# Step 1: Align price and volume to the common benchmark calendar
t_close = raw_data['Close'][trace_ticker].reindex(common_dates)
t_vol   = raw_data['Volume'][trace_ticker].reindex(common_dates)

print(f"\nStep 1 — Data alignment")
print(f"  Total benchmark sessions  : {len(common_dates)}")
print(f"  Sessions with valid close : {t_close.notna().sum()}")
print(f"  Sessions with valid volume: {t_vol.notna().sum()}")

# Step 2: Rolling 20-session baseline volume (lagged by 1 to exclude today)
t_baseline    = t_vol.shift(1).rolling(window=20).mean()
t_is_high_vol = t_vol > 2.0 * t_baseline

# Show an example row from inside the 60-session window
sample_date = last_60_dates[30]   # a mid-window date
print(f"\nStep 2 — High-volume detection (sample session: {sample_date.date()})")
print(f"  Closing price on that date : ${t_close.loc[sample_date]:,.2f}")
print(f"  Volume on that date        : {t_vol.loc[sample_date]:,.0f} shares")
print(f"  20-session baseline volume : {t_baseline.loc[sample_date]:,.0f} shares")
print(f"  Threshold (2× baseline)    : {2.0 * t_baseline.loc[sample_date]:,.0f} shares")
print(f"  High-volume flag           : {t_is_high_vol.loc[sample_date]}")

# Step 3: All qualifying event dates (across full study period)
t_event_dates = t_vol[t_is_high_vol].index
print(f"\nStep 3 — Qualifying high-volume sessions (full period)")
print(f"  Total high-volume sessions : {len(t_event_dates)}")
print(f"  Most recent event date     : {t_event_dates[-1].date()}")

# Step 4: Recency
t_recency = (obs_date - t_event_dates[-1]).days
print(f"\nStep 4 — Recency")
print(f"  Observation date           : {obs_date.date()}")
print(f"  Last high-volume session   : {t_event_dates[-1].date()}")
print(f"  R = {obs_date.date()} − {t_event_dates[-1].date()} = {t_recency} calendar days")

# Step 5: Frequency (60-session window)
t_freq = int(t_is_high_vol.loc[last_60_dates].sum())
print(f"\nStep 5 — Frequency")
print(f"  60-session window              : {last_60_dates[0].date()} → {last_60_dates[-1].date()}")
print(f"  High-volume sessions in window : {t_freq}")
print(f"  F = {t_freq}")

# Step 6: Monetary (60-session window)
t_traded_val = t_close.loc[last_60_dates] * t_vol.loc[last_60_dates]
t_monetary   = float(t_traded_val.sum())
print(f"\nStep 6 — Monetary (traded value)")
print(f"  Formula: sum(Close × Volume) over 60 sessions")
print(f"  Max single-session value   : ${t_traded_val.max():,.2f}")
print(f"  Min single-session value   : ${t_traded_val.min():,.2f}")
print(f"  M = ${t_monetary:,.2f}")
print(f"  log1p(M) [used in scaling] : {np.log1p(t_monetary):.4f}")

# Step 7: Consistency check against the main table
print(f"\nStep 7 — Cross-check against rfm_df")
print(rfm_df.loc[trace_ticker].to_string())
print("=" * 65)


TASK 1.5  WORKED NUMERICAL TRACE — AAPL

Step 1 — Data alignment
  Total benchmark sessions  : 1005
  Sessions with valid close : 1005
  Sessions with valid volume: 1005

Step 2 — High-volume detection (sample session: 2024-11-18)
  Closing price on that date : $228.02
  Volume on that date        : 44,633,700 shares
  20-session baseline volume : 43,871,150 shares
  Threshold (2× baseline)    : 87,742,300 shares
  High-volume flag           : False

Step 3 — Qualifying high-volume sessions (full period)
  Total high-volume sessions : 19
  Most recent event date     : 2024-12-20

Step 4 — Recency
  Observation date           : 2024-12-31
  Last high-volume session   : 2024-12-20
  R = 2024-12-31 − 2024-12-20 = 11 calendar days

Step 5 — Frequency
  60-session window              : 2024-10-07 → 2024-12-31
  High-volume sessions in window : 2
  F = 2

Step 6 — Monetary (traded value)
  Formula: sum(Close × Volume) over 60 sessions
  Max single-session value   : $37,536,079,707.22
  Min 

In [ ]:
assert (t_recency, t_freq) == (rfm_df.loc[trace_ticker,'Recency'], rfm_df.loc[trace_ticker,'Frequency'])

In [ ]:
assert features.isna().sum().sum() == 0, features.isna().sum()

NameError: name 'features' is not defined

In [ ]:
features = risk_return_df[['Mean_Weekly_Return', 'Weekly_Beta']].join(rfm_df, how='inner')
features = features.join(sp500_meta.set_index('Ticker')[['Security', 'GICS Sector']])
features.to_csv("features_task1.csv")
print(f"Final shares entering clustering: {len(features)}")

Final shares entering clustering: 481


In [ ]:
print(exclusion_df.to_string)

<bound method DataFrame.to_string of    Ticker                                             Reason
0     APP              Incomplete history: 935/1005 sessions
1      BE  Extreme daily move detected: 1 day(s) with |re...
2    CVNA  Extreme daily move detected: 5 day(s) with |re...
3    COIN              Incomplete history: 936/1005 sessions
4     CEG              Incomplete history: 742/1005 sessions
5    DXCM  Extreme daily move detected: 1 day(s) with |re...
6     EXE              Incomplete history: 979/1005 sessions
7    FDXF                Incomplete history: 0/1005 sessions
8    FERG                 Excess zero-volume days: 27 (2.7%)
9    GEHC              Incomplete history: 513/1005 sessions
10    GEV              Incomplete history: 193/1005 sessions
11     GL  Extreme daily move detected: 1 day(s) with |re...
12   HONA                Incomplete history: 0/1005 sessions
13   KVUE              Incomplete history: 418/1005 sessions
14      Q                Incomplete history: 0/1

In [ ]:
print(exclusion_df.to_string())

   Ticker                                                                                                            Reason
0     APP                                                                             Incomplete history: 935/1005 sessions
1      BE  Extreme daily move detected: 1 day(s) with |return| > 40% (max: 59.2%). Possible unadjusted split or data error.
2    CVNA  Extreme daily move detected: 5 day(s) with |return| > 40% (max: 56.0%). Possible unadjusted split or data error.
3    COIN                                                                             Incomplete history: 936/1005 sessions
4     CEG                                                                             Incomplete history: 742/1005 sessions
5    DXCM  Extreme daily move detected: 1 day(s) with |return| > 40% (max: 40.7%). Possible unadjusted split or data error.
6     EXE                                                                             Incomplete history: 979/1005 sessions
7    FDX